# 月データでムーンベースの場所を決めよう（データ分析と探究活動）

月の公開データを使って、**月面基地をどこに建てるか**を自分で決めます。
コードを書く必要はありません。セルを上から順に実行し、`# ★ここを変える` と書いてある
数字や名前だけを書き換えて、結果をワークシートに記録していきます。

## あなたのミッションを1つ選ぶ（ワークシートに○）

| ミッション | 基地に必要なこと | 効いてくるデータ |
|---|---|---|
| ☀️ **太陽光発電基地** | よく日が当たること。地面が平らなこと | 太陽高度・温度・（極なら日照率） |
| 🔭 **電波天文台** | 地球の電波が届かない（裏側）。温度が安定 | 地球の仰角・温度の日較差 |
| 🧊 **氷採掘基地** | 氷がありそうなこと（＝永久影のそば） | 永久影までの距離（南極のみ） |
| 🏠 **有人総合基地** | 電力・温度・氷・通信をバランスさせる | ぜんぶ |

**同じデータでも、ミッションが変われば最適な場所は変わります。**
氷採掘は南極に、電波天文台は裏側に、通信重視なら表側に――ミッションによって行き先は
半球ごと変わります。それを today, データで確かめます。

## 進め方
1. 上から順にセルを実行する（Colab なら「ランタイム」→「すべてのセルを実行」）
2. 各ステップで、まずワークシートに **予想** を書く
3. `# ★ここを変える` を書き換えて実行し、結果をワークシートに記録
4. **気づいたこと** を書く

In [ ]:
# 準備：ヘルパー（moonkit）を読み込む
from moonkit import *
for k in DATASETS:
    print(k, ':', len(load(k)), '行')

---
## ステップ1：月の温度は1日でどれくらい変わる？

月には空気（大気）がありません。空気がないと、温度はどうなるでしょう？

**予想をワークシートに書いてから**、下のセルを実行します。

In [ ]:
my_band = (-10, 10)      # ★ここを変える：調べたい緯度の範囲（例：赤道なら (-10, 10)、緯度45度なら (40, 50)）

band = region(load('温度'), lat=my_band)
diurnal_curve(band)                       # 1日の温度変化カーブ
band = daily_swing(band)                  # 各地点の「1日の平均・較差・ばらつき」を計算
summary(band, 't_mean_K', 't_swing_K', 't_std_K')
#   t_mean_K  … 1日の平均温度
#   t_swing_K … 1日の温度差（いちばん暑い時 − いちばん寒い時）
#   t_std_K   … 温度のばらつき（分散の平方根）

**ワークシートに記録**：1日の温度差（`t_swing_K` の平均）は何 K？　地球の砂漠の昼夜差はせいぜい 20〜30 ℃ です。

**気づいたこと**：なぜこんなに差が大きいの？　カーブの形（朝の上がり方と、夕方〜夜の下がり方）は左右対称？

**もうひとつ試す**：`my_band` を `(-90, -80)`（南極のあたり）にして実行してみよう。
カーブがガタガタになり、温度差も小さくなる。これは **①データの限界**（現地時間の刻みが粗い）と
**②物理**（極では太陽が地平線の近くを回るだけで「昼」と「夜」がはっきりしない）の両方による。
極の温度は不確かなので、**極を細かく見るときは温度ではなく日照のデータ（ステップ4b）を使う**。

---
## ステップ2：月の「海」と「陸」で何が違う？

月を見ると、黒っぽく平らな「海（マリア）」と、白っぽくでこぼこの「陸（高地）」があります。
まず、クレーターの分布から「海」を自分で見つけます。

**予想**：クレーターの数は、月面のどこでも同じくらい？　それとも場所によってかたよる？

In [ ]:
# (1) 全体のクレーターの密度を地図で見る → まわりより少ない「帯」を探す
grid_count(load('クレーター'), lat_step=10, lon_step=10)

In [ ]:
# (2) クレーターの位置を月面画像に重ねる → 少ない場所は、画像のどこ？
scatter(load('クレーター'), 'lon', 'lat')

In [ ]:
# (3) 公式の「海」の座標（USGS 地名辞典の23の海）を使って、海と陸のクレーターを数値で比べる
my_scale = 0.8      # ★ここを変える：海の範囲。1.0 で海の縁まで、小さくすると中心部だけ

c = near_maria(load('クレーター'), scale=my_scale)
print(c['区分'].value_counts())               # 海・陸それぞれのクレーターの数
print(summary_by(c, group='区分', value='diam_km'))   # 直径の平均など

age = near_maria(load('クレーター年代'), scale=my_scale)   # 年代（DeepCraters）。数字が大きいほど新しい
print('推定年代の平均：', age.groupby('区分')['Age'].mean().round(2).to_dict())

# --- 答え合わせ：USGS の公式地質図と比べる ---
geo = load('地質')          # 1度グリッド。海/陸 と相対年代 age_index（1古〜5新）
print('USGS 地質図での相対年代の平均（大きいほど新しい）：')
print(summary_by(geo, group='区分', value='age_index')[['件数', '平均']])

import numpy as np
def _sea_fraction(df):
    w = np.cos(np.radians(df['lat']))         # 面積は高緯度ほど小さい
    return float(w[df['区分'] == '海'].sum() / w.sum())
circle = _sea_fraction(near_maria(load('温度')[['lat', 'lon']], scale=my_scale))
usgs = _sea_fraction(geo)
print('海の面積割合：円近似 {:.1%} ／ USGS地質図 {:.1%}（文献値 約16%）'.format(circle, usgs))
print('→ 円で囲むと大きめに出る。なぜ？（海岸線の外の陸も丸に入るから）')

**ワークシートに記録**：海と陸のクレーターの数、直径の平均、年代の平均。

**気づいたこと**：
- 海のほうがクレーターが少ないのはなぜ？（ヒント：海は昔、溶けた溶岩でおおわれて古いクレーターが消えた）
- クレーターが少ない＝新しい、と言えるのはなぜ？

**基地との関係**：海は平らで**着陸しやすい**が若い。高地は古くてでこぼこ。氷は極にしかない。
どれを重く見るかは基地の**目的**しだい――それを次のステップで、月ぜんたいを見ながら決めます。

---
## ステップ3：月ぜんたいで環境を見る

ここまでで「温度」「海と陸」を見ました。基地の場所を決めるには、もう少し指標が要ります。
`load('環境')` は、月ぜんたいを1度マスに区切って、基地選びに関わる指標をまとめた表です。

| 列 | 意味 |
|---|---|
| `temp_amp_K` | 1日の温度差（熱ストレス）。赤道で大・極で小 |
| `night_min_K` | 夜の底冷え。低いほど、夜を越すのに熱が要る |
| `noon_sun_elev_deg` | 正午の太陽高度（= 90 − \|緯度\|）。発電量と熱負荷の代理 |
| `earth_elev_deg` | 地球の仰角。**正＝表側**（通信できる）、**負＝裏側**（地球が見えない＝電波が静か） |
| `slope_deg` | 地面の傾き（LOLA の標高から計算）。小さいほど平ら。**\|緯度\|≥85° は無し**（極は日照データ側で見る） |
| `区分` | 海／陸（おおまかな「平ら／でこぼこ」の目安） |

**予想**：「温度が安定」「地球が見える」「日がよく当たる」「地面が平ら」――この4つが**全部そろう**場所はあると思う？

In [ ]:
env = load('環境')

# 全球マップ：熱ストレスと、地球の見えかた
scatter(env, 'lon', 'lat', color='temp_amp_K')      # 明るいほど1日の温度差が大きい
scatter(env, 'lon', 'lat', color='earth_elev_deg')  # 正（明るい）＝表側 / 負（暗い）＝裏側
scatter(env, 'lon', 'lat', color='slope_deg')       # 明るいほど急。海（マリア）は暗い＝平ら

# 代表的な地域タイプを比べる（load('地域') で名前の一覧が見られる）
for name in ['赤道の海（静かの海）', '中緯度の火砕丘（Aristarchus 高原）',
             '裏側・赤道（電波天文の候補域）', '南極（Shackleton-de Gerlache）']:
    r = region_type(env, name)
    print('{:32s} 日較差 {:3.0f}K   夜の底 {:3.0f}K   太陽高度 {:2.0f}°   地球の仰角 {:+3.0f}°   傾斜 {:4.1f}°'.format(
        name, r['temp_amp_K'].mean(), r['night_min_K'].mean(),
        r['noon_sun_elev_deg'].mean(), r['earth_elev_deg'].mean(), r['slope_deg'].mean()))

**ワークシートに記録**：4つの地域タイプの、日較差・夜の底・太陽高度・地球の仰角。

**気づいたこと**：
- **どこも「全部で一番」にはならない**。赤道の海＝地球が真上・平らだが日較差 300K。
  南極＝日較差は小さいが太陽は低く、地球は地平線すれすれ。裏側＝地球が見えない代わりに電波が静か。
- あなたのミッションにとって、いちばん大事なのはどの列？

---
## ステップ4：地域タイプを選んで、ミッションに合わせて評価する

`load('地域')` の候補地域から**あなたのミッションに合いそうなもの**を1つ選び、その中で
複数の指標を「0〜1の点数」に直して重みをつけ、点数の高い場所を探します（＝あなたのスコア式）。

| ミッション | 選ぶ地域の例 | `want`（重み）の例 |
|---|---|---|
| ☀️ 太陽光発電 | 「赤道の海（静かの海）」と「南極」を両方試す | `{'noon_sun_elev_deg': ('高い', 2), 'temp_amp_K': ('低い', 2), 'slope_deg': ('低い', 1)}` |
| 🔭 電波天文台 | 「裏側・赤道（電波天文の候補域）」 | `{'earth_elev_deg': ('低い', 3), 'temp_amp_K': ('低い', 2), 'slope_deg': ('低い', 1)}` |
| 🧊 氷採掘 | 「南極（Shackleton-de Gerlache）」→ **ステップ4b へ**（永久影は環境データに無い） |  |
| 🏠 有人総合 | いくつか試す（＋ステップ4b） | `{'noon_sun_elev_deg': ('高い', 1), 'temp_amp_K': ('低い', 2), 'earth_elev_deg': ('高い', 1), 'night_min_K': ('高い', 1), 'slope_deg': ('低い', 2)}` |

`earth_elev_deg` を **`('低い', ...)`** にすると「地球が見えない裏側」を高く評価します（電波天文向け）。
`slope_deg` は「地面が平ら（着陸・建設しやすい）」を重く見るとき `('低い', ...)`。**南極を選んだ班は
`slope_deg` が使えない**（極域はステップ4b の LOLA 傾斜で見る）。

In [ ]:
my_region = '赤道の海（静かの海）'    # ★ここを変える：load('地域')['name'] から選ぶ

want = {                              # ★ここを変える：上の表から自分のミッションのものを写す
    'noon_sun_elev_deg': ('高い', 2),
    'temp_amp_K':        ('低い', 2),
    'earth_elev_deg':    ('高い', 0),   # 0 なら気にしない。電波天文なら ('低い', 3)
    'night_min_K':       ('高い', 0),
    'slope_deg':         ('低い', 1),   # 地面の平らさ。南極を選んだ班は 0 にする（極域は使えない）
}

region = region_type(load('環境'), my_region)
best = site_score(region, want, top=10)
print(my_region, 'の中での上位10地点：')
print(best[['lat', 'lon', 'temp_amp_K', 'night_min_K',
            'noon_sun_elev_deg', 'earth_elev_deg', 'slope_deg', 'スコア']].round(1).to_string(index=False))

scatter(site_score(region, want, top=None), 'lon', 'lat', color='スコア')

**ワークシートに記録**：選んだ地域、スコア式（重み）、上位に出た緯度・経度。
そして「**ここに基地を建てる**」と決めた1地点と、その**理由（3つ以上の文で）**。

**気づいたこと**：地域を変えると1位はどう動く？　重みを変えると？
☀️太陽光の人は「赤道の海」と「南極」でスコア上位の値を比べてみよう（明るさ vs 熱の安定）。

---
## ステップ4b：南極を細かく見る（分岐 ― 氷採掘・有人の班だけ）

ここからは班によって分かれます。

- **氷採掘**を選んだ班、または**太陽光・有人でステップ4で「南極」を候補にした**班
  → 別ノートブック **`course_moonbase_polar.ipynb`** を開いて進めます（LOLA の日照率・傾斜・永久影率を使います）。
- **電波天文台**の班、赤道・裏側・中緯度を選んだ班
  → ここはとばして、下の**ステップ5**へ進みます。

永久影（氷のありか）は `load('環境')` には入っていないので、南極を見る班だけが極域専用データに進みます。
「南極が答え」は氷採掘・有人にとっては正当な結論――ミッションが決まれば行き先も決まる、という例です。

---
## ステップ5：他の班・実在の計画と比べる

ここはコード中心の確認のあと、ワークシートに書いてクラスで共有します。

In [ ]:
# 実在の計画は、ミッションによって「半球ごと」場所が違う
programs = [
    ('Artemis III（有人・氷）',      -89.5,  -20.0, '南極'),
    ('LCRT（裏側電波望遠鏡・構想）',   -20.0,  180.0, '裏側'),
    ('Apollo 11（赤道・実績）',        0.67,  23.47, '表側の海'),
    ("Chang'e 4（裏側・実績）",      -45.44, 177.60, '裏側'),
]
env = load('環境')
for name, la, lo, where in programs:
    e = nearest(env, la, lo)
    print('{:26s} 緯度{:6.1f} 経度{:7.1f} [{}]  地球の仰角 {:+.0f}°  日較差 {:.0f}K'.format(
        name, la, lo, where, e['earth_elev_deg'], e['temp_amp_K']))

# Artemis III の南極候補地（着陸地点データから）
ls = load('着陸地点')
print()
print(ls[ls['name'].str.contains('Artemis')][['name', 'lat', 'lon', 'note']].to_string(index=False))


**ワークシートに記録して、クラスで共有**：

- 他の班（ちがうミッション）が選んだ場所は、あなたの場所と**どの半球**にある？　なぜ違った？
- 「1つの正解」はある？　それとも「目的によって最適地は変わる」？
  - 氷採掘 → 南極（永久影に氷）。これは目的が明確なので答えも1つに近い。
  - 電波天文 → 裏側（地球の電波が届かない）。表側では成り立たない。
  - 太陽光・有人 → 南極の尾根と赤道（＋蓄電）で議論が割れる。NASA も候補を1つに絞れていない。
- このデータで「分からないこと」「信じてよいか怪しいこと」は？
  （日照率・傾斜の絶対値／極点の近く／`earth_elev_deg` は秤動を無視／`temp_amp_K` は極で不確か）

---
## （発展・任意）ステップ6：機械が決めた基準と、自分が決めた基準を比べる

ステップ4b で、あなたは重みを決めて「有人基地に向く南極の場所」を選びました。
その「向き・不向き」を機械学習に**当てさせる**とどうなるか。決定木・ニューラルネットなど
5種類のモデルを取り替えて、境界線の形・当たりやすさ・「ルールを説明できるか」を比べます。

このステップは別ノートブック `course_moonbase_ml.ipynb` で行います。時間が余った班・
興味のある人向けで、必須ではありません。

教師なし学習（クラスタリング）や、クレーターの数から絶対年代を出す発展は
`explore_clustering.ipynb` / `explore_advanced.ipynb` にあります。